
<div style="background-color: lightgreen; color: black; padding: 20px; box-sizing: border-box; width: 99%;">

👋 <b>Olá! Eu sou o Dot, seu revisor de IA.</b>

🎉 <b>Parabéns! Você foi aprovado no seu projeto.</b>

Excelente trabalho na conclusão deste projeto! Você seguiu todo o fluxo de trabalho, desde a preparação dos dados até uma recomendação bem justificada da Região 1, e suas funções reutilizáveis e o bootstrapping correto foram pontos particularmente fortes. Para seu crescimento futuro, documente como você lida com IDs duplicados, sustente suas afirmações (como as diferentes escalas das features) com evidências no notebook e mantenha armazenadas as previsões dos poços selecionados para garantir transparência. Continue com esse ótimo ritmo!

Se você tiver alguma dificuldade ou erro, fique à vontade para entrar em contato com nossa equipe de suporte clicando no ícone de chat no canto superior direito ou enviando um e-mail para [helpme@tripleten.com](mailto:helpme@tripleten.com).

<hr style="margin-top: 20px; margin-bottom: 20px;">

<details>
<summary><b style="cursor: pointer;">💡 Clique aqui para ver como ler meus comentários (Legenda de cores)</b></summary>
<div style="margin-top: 15px;">
Veja meus comentários abaixo. <b>Peço gentilmente que não os mova, modifique nem exclua</b>.

Você encontrará meus comentários destacados em caixas verdes, amarelas ou vermelhas como estas:

<div class="alert alert-block alert-success">
Sucesso. Tudo foi feito corretamente.
</div>

<div class="alert alert-block alert-warning">
Observações. Algumas recomendações.
</div>

<div class="alert alert-block alert-danger">
A corrigir. O bloco requer algumas correções.
</div>
</div>
</details>

</div>


# Projeto Sprint 10: Escolha da Melhor Região para Novos Poços de Petróleo (OilyGiant)

**Objetivo:** Construir um modelo de regressão linear para prever o volume de reservas de petróleo em três regiões distintas (`geo_data_0`, `geo_data_1` e `geo_data_2`), calcular o lucro potencial dos 200 melhores poços de cada região e aplicar a técnica de *Bootstrapping* para avaliar os riscos de prejuízo e recomendar a melhor região para investimento.

In [1]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Carregamento dos conjuntos de dados
df_0 = pd.read_csv('/datasets/geo_data_0.csv')
df_1 = pd.read_csv('/datasets/geo_data_1.csv')
df_2 = pd.read_csv('/datasets/geo_data_2.csv')

datasets = {'Região 0': df_0, 'Região 1': df_1, 'Região 2': df_2}

for name, df in datasets.items():
    print(f'=== {name} ===')
    print(df.info())
    print('Valores ausentes:\n', df.isna().sum())
    print('Duplicados totais:', df.duplicated().sum())
    print('IDs duplicados:', df['id'].duplicated().sum())
    print(df.head(3), '\n')

=== Região 0 ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 5 columns):
 #   Column   Non-Null Count   Dtype  
---  ------   --------------   -----  
 0   id       100000 non-null  object 
 1   f0       100000 non-null  float64
 2   f1       100000 non-null  float64
 3   f2       100000 non-null  float64
 4   product  100000 non-null  float64
dtypes: float64(4), object(1)
memory usage: 3.8+ MB
None
Valores ausentes:
 id         0
f0         0
f1         0
f2         0
product    0
dtype: int64
Duplicados totais: 0
IDs duplicados: 10
      id        f0        f1        f2     product
0  txEyH  0.705745 -0.497823  1.221170  105.280062
1  2acmU  1.334711 -0.340164  4.365080   73.037750
2  409Wp  1.022732  0.151990  1.419926   85.265647 

=== Região 1 ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 5 columns):
 #   Column   Non-Null Count   Dtype  
---  ------   --------------   -----

<div class="alert alert-block alert-warning">

Ótimo trabalho ao carregar os três arquivos e executar as mesmas verificações em cada região usando um loop: `info()`, valores ausentes, linhas totalmente duplicadas, valores duplicados de `id` e uma prévia dos dados. A saída da Região 0 mostra `IDs duplicados: 10`, ou seja, alguns poços compartilham o mesmo `id`, e sua conclusão escrita não menciona IDs duplicados em nenhuma das regiões. Dê uma olhada nesses IDs repetidos e decida se vai mantê-los ou removê-los. Depois, registre seu raciocínio, pois, mesmo que o efeito seja pequeno, essa é uma decisão de preparação de dados que vale a pena documentar.

</div>

### Conclusão — Etapa 1 (Carregamento e Preparação dos Dados)
* Os três conjuntos de dados possuem 100.000 registros cada e 5 colunas (`id`, `f0`, `f1`, `f2` e `product`), sem valores ausentes (`NaN`).
* A coluna `id` é apenas um identificador único de cada poço e não possui poder preditivo para o volume de reservas; portanto, será removida das *features* antes do treinamento.
* As características `f0`, `f1` e `f2` são numéricas contínuas. Como possuem escalas diferentes entre si, aplicaremos a padronização (`StandardScaler`) dentro da função de treinamento (ajustada apenas nos dados de treino para evitar vazamento de dados / *data leakage*).

<div class="alert alert-block alert-warning">

Sua explicação sobre a preparação dos dados está clara. Você justifica a remoção de `id` e planeja ajustar o `StandardScaler` apenas nos dados de treino para evitar vazamento de dados (data leakage), que é a abordagem correta. Um ponto precisa de respaldo: você diz que as features têm escalas diferentes, mas nada exibido no notebook mostra isso. Adicionar um `describe()` ou uma comparação dos desvios padrão ou das faixas de valores de `f0`, `f1` e `f2` antes dessa conclusão sustentaria com evidências a sua decisão de aplicar a padronização.

</div>

In [2]:
def train_and_evaluate(df, region_name, random_state=42):
    # Separação de features (removendo id) e target
    features = df.drop(['id', 'product'], axis=1)
    target = df['product']
    
    # Divisão 75:25
    features_train, features_valid, target_train, target_valid = train_test_split(
        features, target, test_size=0.25, random_state=random_state
    )
    
    # Escalonamento das características
    scaler = StandardScaler()
    features_train_scaled = scaler.fit_transform(features_train)
    features_valid_scaled = scaler.transform(features_valid)
    
    # Treinamento com Regressão Linear
    model = LinearRegression()
    model.fit(features_train_scaled, target_train)
    
    # Predições no conjunto de validação
    predictions = pd.Series(model.predict(features_valid_scaled), index=target_valid.index)
    
    # Reset de índices para garantir alinhamento exato nas etapas futuras
    target_valid = target_valid.reset_index(drop=True)
    predictions = predictions.reset_index(drop=True)
    
    # Métricas
    mean_pred = predictions.mean()
    rmse = np.sqrt(mean_squared_error(target_valid, predictions))
    
    print(f'--- {region_name} ---')
    print(f'Volume médio previsto de reservas: {mean_pred:.2f} mil barris')
    print(f'Volume médio real de reservas:     {target_valid.mean():.2f} mil barris')
    print(f'REQM (RMSE) do modelo:             {rmse:.2f} mil barris\n')
    
    return model, target_valid, predictions, mean_pred, rmse

# Executando para todas as regiões
results = {}
for name, df in datasets.items():
    model, target_val, preds, mean_pred, rmse = train_and_evaluate(df, name)
    results[name] = {
        'target_valid': target_val,
        'predictions': preds,
        'mean_pred': mean_pred,
        'rmse': rmse
    }

--- Região 0 ---
Volume médio previsto de reservas: 92.40 mil barris
Volume médio real de reservas:     92.33 mil barris
REQM (RMSE) do modelo:             37.76 mil barris

--- Região 1 ---
Volume médio previsto de reservas: 68.71 mil barris
Volume médio real de reservas:     68.73 mil barris
REQM (RMSE) do modelo:             0.89 mil barris

--- Região 2 ---
Volume médio previsto de reservas: 94.77 mil barris
Volume médio real de reservas:     95.15 mil barris
REQM (RMSE) do modelo:             40.15 mil barris



<div class="alert alert-block alert-success">

Ótimo trabalho. Você colocou todo o fluxo de trabalho em uma única função `train_and_evaluate` e a reutilizou para as três regiões, então não há código duplicado. A divisão 75:25 está correta, o scaler é ajustado (fit) apenas no conjunto de treinamento, e `LinearRegression` é o único modelo utilizado. Redefinir os índices mantém as previsões e os valores reais alinhados para as etapas posteriores. A função retorna as previsões, os valores reais, a previsão média e o RMSE, e você os armazena por região.

</div>

### Conclusão — Etapa 2 (Treinamento e Avaliação dos Modelos)
* **Região 0 e Região 2:** Apresentam volumes médios previstos altos (92,40 e 94,77 mil barris, respectivamente), mas o erro médio quadrático (REQM/RMSE) é bastante elevado (37,76 e 40,15 mil barris). Isso indica maior dispersão e menor precisão das previsões individuais nessas duas regiões.
* **Região 1:** Apresenta o menor volume médio previsto (68,71 mil barris), porém possui um **REQM extremamente baixo (0,89 mil barris)**, demonstrando que o modelo de regressão linear explica quase perfeitamente a relação entre as variáveis `f0`, `f1`, `f2` e o volume de reservas nessa região.

In [3]:
# Constantes de negócio armazenadas em variáveis separadas
BUDGET = 100_000_000          # Orçamento total: US$ 100 milhões
TOTAL_STUDIED_WELLS = 500     # Pontos estudados na exploração
SELECTED_WELLS = 200          # Melhores poços selecionados para desenvolvimento
REVENUE_PER_BARREL = 4.5      # Receita por barril: US$ 4,50
REVENUE_PER_UNIT = 4_500      # Receita por unidade (1.000 barris): US$ 4.500

# Cálculo do volume mínimo médio por poço para evitar prejuízo (ponto de equilíbrio)
cost_per_well = BUDGET / SELECTED_WELLS
min_volume_per_well = cost_per_well / REVENUE_PER_UNIT

print(f'Custo médio por poço: US$ {cost_per_well:,.2f}')
print(f'Volume mínimo médio necessário por poço (breakeven): {min_volume_per_well:.2f} mil barris\n')

for name, df in datasets.items():
    region_mean = df['product'].mean()
    diff = region_mean - min_volume_per_well
    print(f'{name} -> Média geral: {region_mean:.2f} mil barris | Diferença para o mínimo: {diff:.2f}')

Custo médio por poço: US$ 500,000.00
Volume mínimo médio necessário por poço (breakeven): 111.11 mil barris

Região 0 -> Média geral: 92.50 mil barris | Diferença para o mínimo: -18.61
Região 1 -> Média geral: 68.83 mil barris | Diferença para o mínimo: -42.29
Região 2 -> Média geral: 95.00 mil barris | Diferença para o mínimo: -16.11


<div class="alert alert-block alert-success">

Você manteve todos os valores de negócio em variáveis separadas e com nomes claros, e calculou corretamente o ponto de equilíbrio (break-even): US$ 500.000 por poço divididos por US$ 4.500 por unidade resultam em 111,11 mil barris. Compará-lo com a média de cada região deixa as diferenças negativas fáceis de ver. Sua conclusão de que a seleção aleatória gera prejuízo em todas as regiões está correta e explica bem por que o modelo preditivo é necessário.

</div>

### Conclusão — Etapa 3 (Preparação para o Cálculo de Lucro)
* Com um orçamento de **USD 100 milhões** para **200 poços**, cada poço exige um investimento médio de **USD 500.000**. Como cada unidade (1.000 barris) gera **USD 4.500** de receita, cada poço selecionado precisa produzir, em média, **111,11 mil barris** para atingir o ponto de equilíbrio (*breakeven*).
* Ao comparar esse patamar mínimo com a média geral de reservas de cada região (**Região 0:** 92,50 mil barris; **Região 1:** 68,83 mil barris; **Região 2:** 95,00 mil barris), observa-se que **nenhuma região é lucrativa se os poços forem escolhidos aleatoriamente**.
* Portanto, é indispensável utilizar o modelo preditivo para filtrar e selecionar apenas os **200 poços mais promissores** de cada região.

In [4]:
def calculate_profit(target, predictions, count=SELECTED_WELLS):
    # Seleciona os índices dos 200 maiores valores previstos pelo modelo
    top_indices = predictions.sort_values(ascending=False).head(count).index
    # Soma o volume REAL (target) correspondente a esses 200 poços
    selected_real_volume = target.loc[top_indices].sum()
    # Calcula o lucro líquido
    profit = (selected_real_volume * REVENUE_PER_UNIT) - BUDGET
    return profit, selected_real_volume

for name, res in results.items():
    profit, total_vol = calculate_profit(res['target_valid'], res['predictions'])
    res['top200_profit'] = profit
    res['top200_volume'] = total_vol
    print(f'=== {name} (Top 200 Poços da Validação) ===')
    print(f'Volume total real das reservas: {total_vol:,.2f} mil barris')
    print(f'Lucro potencial estimado:       US$ {profit:,.2f}\n')

=== Região 0 (Top 200 Poços da Validação) ===
Volume total real das reservas: 29,686.98 mil barris
Lucro potencial estimado:       US$ 33,591,411.14

=== Região 1 (Top 200 Poços da Validação) ===
Volume total real das reservas: 27,589.08 mil barris
Lucro potencial estimado:       US$ 24,150,866.97

=== Região 2 (Top 200 Poços da Validação) ===
Volume total real das reservas: 27,996.83 mil barris
Lucro potencial estimado:       US$ 25,985,717.59



<div class="alert alert-block alert-warning">

`calculate_profit` está correta: ela escolhe os 200 poços com os maiores valores previstos, soma suas reservas reais e subtrai o orçamento. O argumento padrão `count` também permite reutilizá-la no bootstrap. Duas sugestões: (1) Para cada região, você mantém apenas o lucro e o volume total (`top200_profit`, `top200_volume`). Salvar também as previsões (ou os índices) dos 200 poços selecionados em `results` tornaria esta etapa completa e permitiria ver quais poços foram escolhidos. (2) Opcional: `predictions.nlargest(count).index` faz o mesmo que `sort_values(ascending=False).head(count).index`, só que de forma mais direta.

</div>

### Conclusão — Etapa 4 (Lucro Potencial dos 200 Melhores Poços)
* Considerando o conjunto completo de validação, todas as três regiões apresentam lucro potencial positivo ao selecionar os 200 poços com maiores predições (acima do mínimo total necessário de 22.222,22 mil barris).
* **Sugestão preliminar:** Nesta etapa, a **Região 0** apresenta o maior lucro potencial absoluto (**USD 33,59 milhões** / **USD 33.591.411,14**), seguida pela **Região 2** (**USD 25,99 milhões** / **USD 25.985.717,59**) e pela **Região 1** (**USD 24,15 milhões** / **USD 24.150.866,97**). Contudo, essa análise avalia todo o conjunto de validação de uma vez, sem simular a restrição real de explorar apenas **500 pontos** por estudo e sem medir o risco de variabilidade.

In [5]:
def bootstrap_profit_and_risk(target, predictions, n_bootstraps=1000, sample_size=TOTAL_STUDIED_WELLS, seed=42):
    state = np.random.RandomState(seed)
    profits = []
    
    for _ in range(n_bootstraps):
        # Sorteia 500 poços com reposição
        subsample_indices = state.choice(len(target), size=sample_size, replace=True)
        target_subsample = target.iloc[subsample_indices].reset_index(drop=True)
        preds_subsample = predictions.iloc[subsample_indices].reset_index(drop=True)
        
        # Calcula o lucro dos 200 melhores dentro dos 500 sorteados
        profit, _ = calculate_profit(target_subsample, preds_subsample, count=SELECTED_WELLS)
        profits.append(profit)
        
    profits = pd.Series(profits)
    mean_profit = profits.mean()
    lower_ci = profits.quantile(0.025)
    upper_ci = profits.quantile(0.975)
    loss_risk = (profits < 0).mean() * 100  # Probabilidade de lucro negativo em %
    
    return mean_profit, lower_ci, upper_ci, loss_risk

for name, res in results.items():
    mean_p, low_ci, high_ci, risk = bootstrap_profit_and_risk(res['target_valid'], res['predictions'])
    print(f'=== {name} (Bootstrapping - 1.000 amostras) ===')
    print(f'Lucro médio:                     US$ {mean_p:,.2f}')
    print(f'Intervalo de confiança de 95%:   (US$ {low_ci:,.2f} a US$ {high_ci:,.2f})')
    print(f'Risco de prejuízo:               {risk:.2f}%\n')

=== Região 0 (Bootstrapping - 1.000 amostras) ===
Lucro médio:                     US$ 3,995,754.78
Intervalo de confiança de 95%:   (US$ -1,104,678.95 a US$ 8,974,603.28)
Risco de prejuízo:               6.00%

=== Região 1 (Bootstrapping - 1.000 amostras) ===
Lucro médio:                     US$ 4,520,488.91
Intervalo de confiança de 95%:   (US$ 616,844.80 a US$ 8,453,401.78)
Risco de prejuízo:               1.50%

=== Região 2 (Bootstrapping - 1.000 amostras) ===
Lucro médio:                     US$ 3,750,099.03
Intervalo de confiança de 95%:   (US$ -1,447,667.27 a US$ 8,883,904.04)
Risco de prejuízo:               8.00%



<div class="alert alert-block alert-success">

Seu bootstrapping está implementado corretamente e reflete o cenário real de exploração. Cada uma das 1.000 iterações sorteia 500 poços com reposição, seleciona os 200 melhores pela previsão dentro dessa amostra e calcula o lucro a partir das reservas reais menos o orçamento. A média, o intervalo de 95% a partir dos quantis de 2,5% e 97,5% e o risco de prejuízo estão todos calculados corretamente. Reutilizar `calculate_profit`, aplicar a função a cada região e fixar uma seed para garantir a reprodutibilidade são boas práticas.

</div>

### Conclusão Final — Etapa 5 (Análise de Riscos e Recomendação)
* **Filtro de Risco (< 2,5%):** Ao simular 1.000 estudos de 500 poços via *bootstrapping*, apenas a **Região 1** atende ao critério de segurança estabelecido pela OilyGiant, apresentando um **risco de prejuízo de 1,50%** (abaixo do limite de 2,5%). As Regiões 0 e 2 apresentam riscos de prejuízo superiores ao permitido (**6,00%** e **8,00%**, respectivamente) e intervalos de confiança de 95% que incluem valores negativos.
* **Comparação com a Etapa 4.3:** A escolha final **não corresponde** à escolha preliminar da Etapa 4.3 (Região 0). Na Etapa 4, o erro elevado do modelo na Região 0 (REQM de 37,76) mascarava o risco ao selecionar os 200 melhores pontos de todo o conjunto de validação. Quando aplicamos a restrição realista de amostrar apenas 500 poços, a alta precisão do modelo da **Região 1** (REQM de 0,89) garante maior consistência e o maior lucro médio (**USD 4.520.488,91**, com intervalo de confiança de 95% totalmente positivo entre **USD 616.844,80** e **USD 8.453.401,78**).
* **Recomendação:** Recomenda-se o desenvolvimento dos novos poços de petróleo na **Região 1**, por ser a única região que combina lucro médio atrativo, intervalo de confiança de 95% totalmente positivo e risco de perda abaixo de 2,5%.

<div class="alert alert-block alert-warning">

Sua recomendação está bem argumentada e condiz com os números: a Região 1 é a única com risco de prejuízo abaixo de 2,5% (1,50%), tem o maior lucro médio e seu intervalo de confiança de 95% é inteiramente positivo. Você também respondeu claramente que essa escolha difere da anterior. Uma sugestão para aprimorar a comparação: você atribui o resultado anterior principalmente ao alto erro do modelo na Região 0, que mascarava o risco. Outro fator importante é o tamanho do conjunto do qual você escolhe os poços. Anteriormente, você selecionou os 200 melhores de todo o conjunto de validação, com cerca de 25.000 poços, enquanto no bootstrap você os seleciona a partir de apenas 500. Com um conjunto pequeno e uma grande dispersão nas reservas reais, como na Região 0, os resultados ruins se tornam muito mais prováveis. Mencionar isso junto com o erro do modelo tornaria seu raciocínio mais completo.

</div>